In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U timm kagglehub

import os
GPU_OK = "A100" in os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read()
print("A100:", GPU_OK, "| якщо False — конфіг нижче все одно спрацює, просто повільніше")

NVIDIA A100-SXM4-40GB, 40960 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 34.9 MB/s eta 0:00:00
A100: True | якщо False — конфіг нижче все одно спрацює, просто повільніше


In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


## Дані

In [ ]:
DATA_DIR = kagglehub.competition_download("birdclef-2026")
print("DATA_DIR:", DATA_DIR)
print("train_audio:", os.path.isdir(f"{DATA_DIR}/train_audio"),
      "| soundscapes:", os.path.isdir(f"{DATA_DIR}/train_soundscapes"))

100%|██████████| 15.0G/15.0G [01:55<00:00, 139MB/s]

Extracting files...


DATA_DIR: /root/.cache/kagglehub/competitions/birdclef-2026
train_audio: True | soundscapes: True


# Таргети перча

In [ ]:
import time

NB_TEACHER = "denyspushkaruks/notebook5c8d3d54bf"

def fetch(fn, tries=3):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            time.sleep(5 * (i + 1))

TEACHER_DIR = fetch(lambda: kagglehub.notebook_output_download(NB_TEACHER))

for dp, _, fn in os.walk(TEACHER_DIR):
    if "perch_probs.npy" in fn: TEACHER_DIR = dp; break

need = ["perch_probs.npy", "perch_mask.npy", "perch_index.csv", "species_list.txt"]
print("TEACHER_DIR:", TEACHER_DIR)
print({f: os.path.exists(f"{TEACHER_DIR}/{f}") for f in need})

Extracting files...
TEACHER_DIR: /root/.cache/kagglehub/notebooks/denyspushkaruks/notebook5c8d3d54bf/output/versions/1
{'perch_probs.npy': True, 'perch_mask.npy': True, 'perch_index.csv': True, 'species_list.txt': True}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/birdclef2026"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Чекпойнти:", OUTPUT_DIR)

Mounted at /content/drive
Чекпойнти: /content/drive/MyDrive/birdclef2026


# Knowledge Distillation

In [ ]:
import os, ast, math, random, time, gc, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torchaudio, timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")


class CFG:
    seed = 42
    data_dir = "/kaggle/input/competitions/birdclef-2026"
    output_dir = "/kaggle/working"
    amp_dtype = "fp16"
    channels_last = False
    tf32 = False
    cudnn_benchmark = True


    sr = 32000; duration = 15; n_samples = sr * duration
    window_sec = 5; window_samples = sr * 5
    n_fft = 2048; hop_length = 512; n_mels = 128; fmin = 20; fmax = 16000

    model_name = "tf_efficientnet_b0.ns_jft_in1k"; in_channels = 1; pretrained = True

    use_rnn = True
    rnn_hidden = 256
    rnn_layers = 1
    rnn_dropout = 0.2

    use_rnn = True
    rnn_freq_pool = True
    rnn_hidden = 128
    ckpt_tag = "kd_rnn2"

    epochs = 10; batch_size = 32; lr = 1e-3; weight_decay = 1e-5
    warmup_epochs = 2; n_folds = 3
    num_workers = 2; prefetch_factor = 2

    mixup_alpha = 0.5; mixup_prob = 0.6
    freq_mask = 20; time_mask = 80; noise = 0.005

    clip_loss_weight = 1.0; frame_loss_weight = 0.5

    teacher_dir = "/kaggle/working/perch"
    hard_w = 1.0
    kd_alpha = 1.0
    kd_T = 2.0
    emb_kd_w = 0.0
    emb_dim = 1536

    teacher_max_sec = 60
    crop_in_teacher = True

    use_secondary = True; sec_weight = 0.5; label_smooth = 0.01
    min_rating = 2.0; use_soundscapes = True

    resume = False; resume_lr = 3e-4


AMP = {"fp16": torch.float16, "bf16": torch.bfloat16}


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); torch.backends.cudnn.benchmark = True


def channels_last_ok(cfg, dev):
    if not cfg.channels_last or not torch.cuda.is_available(): return False
    try:
        for conv, x in [
            (nn.Conv2d(1, 32, 3, stride=2, padding=1), torch.randn(2, 1, 64, 64)),
            (nn.Conv2d(32, 32, 3, padding=1, groups=32), torch.randn(2, 32, 64, 64)),
        ]:
            conv = conv.to(dev).to(memory_format=torch.channels_last)
            x = x.to(dev).contiguous(memory_format=torch.channels_last)
            with autocast(dtype=AMP[cfg.amp_dtype]):
                conv(x).sum().backward()
        return True
    except RuntimeError as e:
        print(f"  channels_last ВИМКНЕНО: cuDNN не тягне ({str(e).splitlines()[0][:70]})")
        return False


def setup_hw(cfg):
    if cfg.tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = cfg.cudnn_benchmark
    cfg.channels_last = channels_last_ok(cfg, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} | amp={cfg.amp_dtype} "
              f"| channels_last={cfg.channels_last} | tf32={cfg.tf32}")

## Teacher

In [ ]:
class Teacher:
    def __init__(self, tdir, species, cfg):
        self.P = np.load(os.path.join(tdir, "perch_probs.npy"), mmap_mode="r")
        self.mask = torch.from_numpy(np.load(os.path.join(tdir, "perch_mask.npy")))
        self.win = cfg.window_sec
        self.E = None
        if cfg.emb_kd_w > 0:
            self.E = np.load(os.path.join(tdir, "perch_emb.npy"), mmap_mode="r")

        tsp = open(os.path.join(tdir, "species_list.txt")).read().split("\n")
        assert tsp == species, "порядок видів у вчителя і в трені мусить збігатись"

        ix = pd.read_csv(os.path.join(tdir, "perch_index.csv"))
        ix["row"] = np.arange(len(ix))
        agg = ix.groupby("filename", sort=False)["row"].agg(["min", "count"])
        self.loc = {f: (int(r["min"]), int(r["count"])) for f, r in agg.iterrows()}
        print(f"Учитель: {self.P.shape[0]} вікон | {len(self.loc)} файлів | "
              f"{int(self.mask.sum())}/{len(species)} видів під KD")

    def _rows(self, key, off_sec, dur_sec):
        loc = self.loc.get(key)
        if loc is None: return None
        s, n = loc
        a = int(off_sec // self.win)
        if a >= n: return None
        b = min(n, max(a + 1, math.ceil((off_sec + dur_sec) / self.win)))
        return s + a, s + b

    def probs(self, key, off_sec, dur_sec):
        r = self._rows(key, off_sec, dur_sec)
        if r is None: return None
        return self.P[r[0]:r[1]].astype(np.float32).max(0)

    def emb(self, key, off_sec, dur_sec):
        r = self._rows(key, off_sec, dur_sec)
        if r is None or self.E is None: return None
        return self.E[r[0]:r[1]].astype(np.float32).mean(0)

## Датасет

In [ ]:
class BirdDS(Dataset):
    def __init__(self, df, sp2i, cfg, mode="train", teacher=None):
        self.df = df.reset_index(drop=True)
        self.sp2i = sp2i; self.nc = len(sp2i); self.cfg = cfg
        self.mode = mode; self.teacher = teacher

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        ss = r.get("start_sec", None)
        if isinstance(ss, float) and np.isnan(ss): ss = None
        w, off = self._load(r["filepath"], ss)
        if self.mode == "train":
            if random.random() < .5: w = w + torch.randn_like(w) * self.cfg.noise
            if random.random() < .5: w = w * random.uniform(0.7, 1.3)

        t = np.zeros(self.nc, np.float32); h = 0.0
        e = np.zeros(self.cfg.emb_dim if self.cfg.emb_kd_w > 0 else 1, np.float32)
        if self.teacher is not None:
            tp = self.teacher.probs(r["tkey"], off, self.cfg.duration)
            if tp is not None:
                t = tp; h = 1.0
                if self.cfg.emb_kd_w > 0:
                    te = self.teacher.emb(r["tkey"], off, self.cfg.duration)
                    if te is not None: e = te
        return w, self._lbl(r), torch.from_numpy(t), torch.tensor(h), torch.from_numpy(e)

    def _load(self, fp, ss):
        c = self.cfg
        try:
            info = torchaudio.info(fp)
            sr0, total = info.sample_rate, info.num_frames
            need = int(c.duration * sr0)

            if ss is not None:
                s0 = int(float(ss) * sr0)
            elif self.mode == "train":
                hi = max(0, total - need)
                if c.crop_in_teacher:
                    hi = min(hi, max(0, int(c.teacher_max_sec * sr0) - need))
                s0 = random.randint(0, hi) if hi > 0 else 0
            else:
                s0 = max(0, total - need) // 2

            w, sr = torchaudio.load(fp, frame_offset=s0, num_frames=need)
            if w.shape[0] > 1: w = w.mean(0, keepdim=True)
            w = w.squeeze(0)
            if sr != c.sr: w = torchaudio.functional.resample(w, sr, c.sr)
        except Exception:
            return torch.zeros(c.n_samples), (0.0 if ss is None else float(ss))

        if w.shape[0] < c.n_samples:
            w = F.pad(w, (0, c.n_samples - w.shape[0]))
        w = w[:c.n_samples]
        off = s0 / sr0
        return w, off

    def _lbl(self, r):
        y = np.zeros(self.nc, np.float32)
        if isinstance(r["primary_label"], str):
            for p in r["primary_label"].split(";"):
                p = p.strip()
                if p in self.sp2i: y[self.sp2i[p]] = 1.0
        if self.cfg.use_secondary and "secondary_labels" in r.index:
            s = r["secondary_labels"]
            if isinstance(s, str) and s not in ("", "[]"):
                try: sl = ast.literal_eval(s)
                except Exception: sl = [x.strip() for x in s.split(";") if x.strip()]
                for x in sl:
                    if x in self.sp2i: y[self.sp2i[x]] = self.cfg.sec_weight
        return torch.from_numpy(y)

## Модель і лосси

In [ ]:
class GPUMel(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            c.sr, c.n_fft, hop_length=c.hop_length, n_mels=c.n_mels,
            f_min=c.fmin, f_max=c.fmax, power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        m = self.db(self.mel(x))
        m = (m - m.mean((-2, -1), keepdim=True)) / (m.std((-2, -1), keepdim=True) + 1e-6)
        return m.unsqueeze(1)


def spec_aug(m, fm=20, tm=80):
    B, C, Fq, T = m.shape
    for i in range(B):
        if random.random() < .5:
            f = random.randint(0, fm); f0 = random.randint(0, max(0, Fq - f)); m[i, :, f0:f0 + f, :] = 0
        if random.random() < .5:
            t = random.randint(0, tm); t0 = random.randint(0, max(0, T - t)); m[i, :, :, t0:t0 + t] = 0
    return m


def wave_mixup(x, y, t, h, e, alpha=0.5):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mix = lambda a: lam * a + (1 - lam) * a[idx]
    return mix(x), mix(y), mix(t), mix(h), mix(e)


class LSEPool(nn.Module):
    def __init__(self, r=5.0):
        super().__init__()
        self.r = nn.Parameter(torch.tensor(r))
    def forward(self, x):
        return torch.logsumexp(self.r * x, dim=1) / self.r


class SEDModel(nn.Module):
    def __init__(self, cfg, nc):
        super().__init__()
        self.backbone = timm.create_model(cfg.model_name, pretrained=cfg.pretrained,
                                          in_chans=cfg.in_channels, num_classes=0, global_pool="")
        with torch.no_grad():
            o = self.backbone(torch.randn(1, cfg.in_channels, cfg.n_mels, 938))
            if o.dim() == 4: _, c, h, _ = o.shape; self.fd = c * h; self.ch = c
            else: self.fd = o.shape[-1]; self.ch = self.fd

        self.freq_pool = getattr(cfg, "rnn_freq_pool", False) and getattr(cfg, "use_rnn", False)
        d = self.ch if self.freq_pool else self.fd

        self.rnn = None
        if getattr(cfg, "use_rnn", False):
            self.rnn = nn.GRU(d, cfg.rnn_hidden, cfg.rnn_layers, batch_first=True,
                              bidirectional=True,
                              dropout=cfg.rnn_dropout if cfg.rnn_layers > 1 else 0.)
            d = cfg.rnn_hidden * 2

        self.fc = nn.Linear(d, nc)
        self.lse = LSEPool(r=5.0)
        self.drop = nn.Dropout(0.3)
        self.proj = nn.Linear(d, cfg.emb_dim) if cfg.emb_kd_w > 0 else None

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            b, c, h, w = f.shape
            if self.freq_pool:
                f = f.mean(dim=2).permute(0, 2, 1)
            else:
                f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)

        if self.rnn is not None:
            f, _ = self.rnn(f)
        f = self.drop(f)

        framewise = self.fc(f)
        clipwise = self.lse(framewise)
        framewise_max = framewise.max(dim=1).values
        emb = self.proj(f.mean(1)) if self.proj is not None else None
        return clipwise, framewise_max, emb


class SEDLoss(nn.Module):
    def __init__(self, clip_w=1.0, frame_w=0.5, ls=0.01):
        super().__init__()
        self.clip_w = clip_w; self.frame_w = frame_w; self.ls = ls
    def forward(self, clipwise, framewise_max, target):
        if self.ls > 0:
            target = target * (1 - self.ls) + self.ls / 2
        return (self.clip_w * F.binary_cross_entropy_with_logits(clipwise, target) +
                self.frame_w * F.binary_cross_entropy_with_logits(framewise_max, target))


def kd_bce(student_logits, teacher_prob, w, T):
    t = teacher_prob.clamp(1e-6, 1 - 1e-6)
    t_logit = torch.log(t) - torch.log1p(-t)
    soft = torch.sigmoid(t_logit / T)
    l = F.binary_cross_entropy_with_logits(student_logits / T, soft, reduction="none")
    return (l * w).sum() / w.sum().clamp(min=1.0) * (T * T)


def pcmap(yt, yp, pad=5):
    aps = []
    for i in range(yt.shape[1]):
        t = np.concatenate([yt[:, i], np.ones(pad)])
        p = np.concatenate([yp[:, i], np.ones(pad)])
        try: aps.append(average_precision_score(t, p))
        except Exception: aps.append(0.)
    return float(np.mean(aps))


def cosine_sched(opt, wu, total):
    def f(s):
        if s < wu: return s / max(1, wu)
        p = (s - wu) / max(1, total - wu)
        return max(0, .5 * (1 + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

## Трен

In [ ]:
def train_ep(model, mel_fn, dl, opt, sched, crit, dev, cfg, cmask):
    model.train()
    sc = GradScaler(enabled=(cfg.amp_dtype == "fp16")); tl = kdl = 0.
    for w, tg, tp, h, te in tqdm(dl, leave=False, desc="train"):
        w, tg, tp = w.to(dev, non_blocking=True), tg.to(dev, non_blocking=True), tp.to(dev, non_blocking=True)
        h, te = h.to(dev, non_blocking=True), te.to(dev, non_blocking=True)
        if random.random() < cfg.mixup_prob:
            w, tg, tp, h, te = wave_mixup(w, tg, tp, h, te, cfg.mixup_alpha)
        with torch.no_grad(): m = mel_fn(w)
        m = spec_aug(m, cfg.freq_mask, cfg.time_mask)
        if cfg.channels_last: m = m.contiguous(memory_format=torch.channels_last)

        opt.zero_grad(set_to_none=True)
        with autocast(dtype=AMP[cfg.amp_dtype]):
            clip, frame_max, emb = model(m)
            loss = cfg.hard_w * crit(clip, frame_max, tg)

            W = cmask.unsqueeze(0) * h.unsqueeze(1)
            kd = kd_bce(clip, tp, W, cfg.kd_T) + cfg.frame_loss_weight * kd_bce(frame_max, tp, W, cfg.kd_T)
            loss = loss + cfg.kd_alpha * kd

            if cfg.emb_kd_w > 0:
                cos = 1 - F.cosine_similarity(emb.float(), te, dim=1)
                loss = loss + cfg.emb_kd_w * (cos * h).sum() / h.sum().clamp(min=1.0)

        sc.scale(loss).backward(); sc.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        sc.step(opt); sc.update(); sched.step()
        tl += loss.item(); kdl += kd.item()
    return tl / len(dl), kdl / len(dl)


@torch.no_grad()
def val_ep(model, mel_fn, dl, crit, dev, cfg):
    model.eval(); ps, ts, tl = [], [], 0.
    for w, tg, *_ in dl:
        w, tg = w.to(dev), tg.to(dev)
        m = mel_fn(w)
        if cfg.channels_last: m = m.contiguous(memory_format=torch.channels_last)
        with autocast(dtype=AMP[cfg.amp_dtype]):
            clip, frame_max, _ = model(m)
            tl += crit(clip, frame_max, tg).item()
        logits = 0.5 * clip + 0.5 * frame_max
        ps.append(torch.sigmoid(logits).cpu().numpy()); ts.append(tg.cpu().numpy())
    ps = np.concatenate(ps); ts = (np.concatenate(ts) > .5).astype(int)
    return tl / len(dl), pcmap(ts, ps)

In [ ]:
def prep(data_dir, cfg):
    tax = os.path.join(data_dir, "taxonomy.csv")
    sp = sorted(pd.read_csv(tax)["primary_label"].unique().tolist()) if os.path.exists(tax) else None

    df = pd.read_csv(os.path.join(data_dir, "train.csv"))
    adir = os.path.join(data_dir, "train_audio")
    df["filepath"] = df["filename"].apply(lambda x: os.path.join(adir, x))
    df["tkey"] = df["filename"].apply(lambda x: "train_audio/" + str(x).replace("\\", "/"))
    df["start_sec"] = np.nan; df["source"] = "audio"
    if cfg.min_rating > 0 and "rating" in df.columns:
        df = df[(df["rating"] >= cfg.min_rating) | (df["rating"] == 0)].reset_index(drop=True)
    if sp is None: sp = sorted(df["primary_label"].unique().tolist())

    tsl = os.path.join(data_dir, "train_soundscapes_labels.csv")
    tsd = os.path.join(data_dir, "train_soundscapes")
    if cfg.use_soundscapes and os.path.exists(tsl) and os.path.isdir(tsd):
        ts = pd.read_csv(tsl)
        ts["filepath"] = ts["filename"].apply(lambda x: os.path.join(tsd, x))
        ts["tkey"] = ts["filename"].apply(lambda x: "train_soundscapes/" + os.path.basename(str(x)))
        def _ps(v):
            if isinstance(v, (int, float)): return float(v)
            s = str(v).strip(); pts = s.split(":")
            try:
                if len(pts) == 3: return int(pts[0]) * 3600 + int(pts[1]) * 60 + float(pts[2])
                if len(pts) == 2: return int(pts[0]) * 60 + float(pts[1])
                return float(s)
            except Exception: return 0.
        ts["start_sec"] = ts["start"].apply(_ps); ts["source"] = "sc"
        if "secondary_labels" not in ts.columns: ts["secondary_labels"] = ""
        cols = ["filepath", "tkey", "primary_label", "secondary_labels", "start_sec", "source"]
        if "secondary_labels" not in df.columns: df["secondary_labels"] = ""
        df = pd.concat([df[cols], ts[cols]], ignore_index=True)
    elif "secondary_labels" not in df.columns:
        df["secondary_labels"] = ""

    print(f"{len(sp)} classes | {len(df)} samples")
    return df, sp

In [ ]:
def train(cfg=None, data_dir=None, output_dir=None, folds=(0, 1, 2)):
    cfg = cfg or CFG()
    data_dir = data_dir or cfg.data_dir
    output_dir = output_dir or cfg.output_dir
    set_seed(cfg.seed); setup_hw(cfg); os.makedirs(output_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {dev} | Model: {cfg.model_name} | KD: alpha={cfg.kd_alpha} T={cfg.kd_T} "
          f"hard_w={cfg.hard_w} emb_w={cfg.emb_kd_w}")

    df, sp = prep(data_dir, cfg); nc = len(sp)
    sp2i = {s: i for i, s in enumerate(sp)}
    with open(os.path.join(output_dir, "species_list.txt"), "w") as f:
        f.write("\n".join(sp))

    teacher = Teacher(cfg.teacher_dir, sp, cfg)
    cmask = teacher.mask.to(dev)
    mel_fn = GPUMel(cfg).to(dev)

    cov = float(df["tkey"].isin(teacher.loc).mean())
    print(f"Покриття вчителем: {cov:.1%} рядків | KD по {int(teacher.mask.sum())}/{nc} видах")
    assert cov > 0.5, f"Учитель бачить лише {cov:.1%} файлів — перевір teacher_dir і tkey"

    df["_s"] = df["primary_label"].apply(lambda x: x.split(";")[0].strip() if isinstance(x, str) else "unk")
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)
    df["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df, df["_s"])): df.loc[vi, "fold"] = fi

    t0 = time.time()
    for fold in folds:
        print(f'\n{"=" * 50}\nFOLD {fold}\n{"=" * 50}')
        tdf = df[df["fold"] != fold].reset_index(drop=True)
        vdf = df[df["fold"] == fold].reset_index(drop=True)
        dkw = dict(num_workers=cfg.num_workers, pin_memory=True,
                   persistent_workers=cfg.num_workers > 0,
                   prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None)
        tdl = DataLoader(BirdDS(tdf, sp2i, cfg, "train", teacher), cfg.batch_size, shuffle=True,
                         drop_last=True, **dkw)
        vdl = DataLoader(BirdDS(vdf, sp2i, cfg, "val", teacher), cfg.batch_size * 2, **dkw)
        print(f"Train: {len(tdf)} ({len(tdl)} batches) | Val: {len(vdf)}")

        model = SEDModel(cfg, nc).to(dev)
        if cfg.channels_last: model = model.to(memory_format=torch.channels_last)

        ckpt_path = os.path.join(output_dir, f"{cfg.ckpt_tag}_fold_{fold}_best.pth")
        start_ep, best, bep, pat = 0, 0., 0, 0
        if cfg.resume and os.path.exists(ckpt_path):
            ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
            model.load_state_dict(ck["model_state_dict"])
            best = ck.get("score", 0.); start_ep = ck.get("epoch", 0)
            print(f"  Resumed from ep{start_ep}, pcmAP={best:.4f}")

        if cfg.resume and start_ep > 0:
            actual_lr, remaining, warmup_ep = cfg.resume_lr, cfg.epochs - start_ep, 0
        else:
            actual_lr, remaining, warmup_ep = cfg.lr, cfg.epochs, cfg.warmup_epochs
        opt = torch.optim.AdamW(model.parameters(), lr=actual_lr, weight_decay=cfg.weight_decay)
        sched = cosine_sched(opt, len(tdl) * warmup_ep, len(tdl) * remaining)
        crit = SEDLoss(cfg.clip_loss_weight, cfg.frame_loss_weight, cfg.label_smooth)

        for ep in range(start_ep, cfg.epochs):
            tl, kdl = train_ep(model, mel_fn, tdl, opt, sched, crit, dev, cfg, cmask)
            vl, vs = val_ep(model, mel_fn, vdl, crit, dev, cfg)
            print(f"Ep {ep + 1}/{cfg.epochs} | TL:{tl:.4f} (KD:{kdl:.4f}) VL:{vl:.4f} "
                  f"pcmAP:{vs:.4f} | {(time.time()-t0)/60:.0f}min")
            if vs > best:
                best, bep, pat = vs, ep + 1, 0
                torch.save({"model_state_dict": model.state_dict(), "score": best, "epoch": bep,
                            "cfg": {"model_name": cfg.model_name, "n_mels": cfg.n_mels, "sr": cfg.sr,
                                    "duration": cfg.duration, "n_fft": cfg.n_fft, "hop_length": cfg.hop_length,
                                    "fmin": cfg.fmin, "fmax": cfg.fmax, "num_classes": nc,
                                    "in_channels": cfg.in_channels, "window_sec": cfg.window_sec,
                                    "use_rnn": cfg.use_rnn, "rnn_hidden": cfg.rnn_hidden,
                                    "rnn_layers": cfg.rnn_layers, "emb_kd_w": cfg.emb_kd_w,
                                    "emb_dim": cfg.emb_dim}, "rnn_freq_pool": cfg.rnn_freq_pool},
                           ckpt_path)
                print(f"  -> Saved ({best:.4f})")
            else:
                pat += 1
                if pat >= 7: print("  Early stop"); break
        print(f"Fold {fold}: ep{bep} pcmAP={best:.4f}")
        del model, opt, sched; gc.collect(); torch.cuda.empty_cache()

    print(f"\nDone! Total: {(time.time() - t0) / 60:.1f}min")

## Конфіг під A100 

In [ ]:
cfg = CFG()
cfg.data_dir = DATA_DIR
cfg.output_dir = OUTPUT_DIR
cfg.teacher_dir = TEACHER_DIR

cfg.use_rnn = True
cfg.rnn_freq_pool = True
cfg.rnn_hidden = 128
cfg.ckpt_tag = "kd_rnn2"

cfg.amp_dtype = "bf16"
cfg.channels_last = True
cfg.tf32 = True
cfg.batch_size = 96
cfg.lr = 3e-3
cfg.num_workers = min(8, os.cpu_count())
cfg.prefetch_factor = 2

cfg.epochs = 10

print(f"batch={cfg.batch_size} lr={cfg.lr} workers={cfg.num_workers} epochs={cfg.epochs}")

batch=96 lr=0.003 workers=8 epochs=10


In [ ]:
!pip -q install soundfile
import soundfile as sf, torch, random, torchaudio

def _load(self, fp, ss):
    c = self.cfg
    try:
        i = sf.info(fp)
        sr0, total = i.samplerate, i.frames
        need = int(c.duration * sr0)

        if ss is not None:
            s0 = int(float(ss) * sr0)
        elif self.mode == "train":
            hi = max(0, total - need)
            if c.crop_in_teacher:
                hi = min(hi, max(0, int(c.teacher_max_sec * sr0) - need))
            s0 = random.randint(0, hi) if hi > 0 else 0
        else:
            s0 = max(0, total - need) // 2

        x, sr = sf.read(fp, start=s0, frames=need, dtype="float32", always_2d=True)
        w = torch.from_numpy(x.mean(axis=1))
        if sr != c.sr: w = torchaudio.functional.resample(w, sr, c.sr)
    except Exception as e:
        print("DECODE FAIL:", fp, type(e).__name__, e)
        return torch.zeros(c.n_samples), (0.0 if ss is None else float(ss))

    if w.shape[0] < c.n_samples:
        w = torch.nn.functional.pad(w, (0, c.n_samples - w.shape[0]))
    return w[:c.n_samples], s0 / sr0

BirdDS._load = _load

df_, _sp = prep(cfg.data_dir, cfg)
for fp in df_.filepath.head(3):
    i = sf.info(fp)
    x, sr = sf.read(fp, start=0, frames=32000 * 15, dtype="float32", always_2d=True)
    print(f"{i.samplerate}Hz {i.frames}f | mean|w|={abs(x).mean():.4f} | НЕ тиша: {abs(x).mean() > 1e-4}")

234 classes | 36738 samples
32000Hz 576768f | mean|w|=0.0108 | НЕ тиша: True
32000Hz 908544f | mean|w|=0.0074 | НЕ тиша: True
32000Hz 2560512f | mean|w|=0.0087 | НЕ тиша: True


In [ ]:
train(cfg, folds=(0, 1, 2))

GPU: NVIDIA A100-SXM4-40GB | amp=bf16 | channels_last=True | tf32=True
Device: cuda | Model: tf_efficientnet_b0.ns_jft_in1k | KD: alpha=1.0 T=2.0 hard_w=1.0 emb_w=0.0
234 classes | 36738 samples
Учитель: 330005 вікон | 45918 файлів | 203/234 видів під KD
Покриття вчителем: 100.0% рядків | KD по 203/234 видах

FOLD 0
Train: 24492 (255 batches) | Val: 12246


train:   0%|          | 0/255 [00:00<?, ?it/s]

Ep 1/10 | TL:4.9241 (KD:4.0253) VL:0.6534 pcmAP:0.2469 | 4min
  -> Saved (0.2469)


train:   0%|          | 0/255 [00:00<?, ?it/s]

Ep 2/10 | TL:4.7730 (KD:4.0011) VL:0.6928 pcmAP:0.4585 | 6min
  -> Saved (0.4585)


...

Ep 8/10 | TL:4.7454 (KD:3.9772) VL:0.7318 pcmAP:0.8074 | 61min
  -> Saved (0.8074)


train:   0%|          | 0/255 [00:00<?, ?it/s]

Ep 9/10 | TL:4.7445 (KD:3.9768) VL:0.7283 pcmAP:0.8153 | 63min
  -> Saved (0.8153)


train:   0%|          | 0/255 [00:00<?, ?it/s]

Ep 10/10 | TL:4.7440 (KD:3.9763) VL:0.7232 pcmAP:0.8135 | 65min
Fold 2: ep9 pcmAP=0.8153

Done! Total: 65.4min
